# DLRM — Deep Learning Recommendation Model
**Stage: Re-Ranking** (fine-grained scoring of candidates from Two-Tower retrieval)

Based on the Databricks DLRM reference — ported to run free on **Google Colab T4** or **Kaggle P100**.
No Spark, no Databricks, no cloud storage required.

## Architecture
```
Dense features  ─► Bottom MLP ──────────────────────────────────┐
(13 continuous)    [in→64→16]                                   │
                                                                 ├─► Feature Interaction ─► Top MLP ─► P(click)
Sparse features ─► Embedding Tables ─► [emb_1, emb_2, …, emb_6]┘
(6 categorical)    (user_id, item_id,
                    category, brand,
                    country, device)
```
**Paper:** Naumov et al. 2019 — 'DLRM: Deep Learning Recommendation Model'
**Used by:** Meta, Twitter, Alibaba

## Stack (all pinned to latest stable, April 2026)
| Databricks Original | This Notebook |
|---|---|
| TorchDistributor + TorchRec | Standard PyTorch (no PySpark needed) |
| Mosaic StreamingDataset (S3) | torch DataLoader |
| Synthetic Delta Table | NumPy-generated synthetic data (same schema) |
| Databricks MLflow | mlflow 3.11.1 (local) |
| dbutils / spark.sql | Removed |

## Free GPU Options
- **Google Colab**: Runtime → Change runtime type → T4 GPU
- **Kaggle**: Settings → Accelerator → GPU (30 hrs/week free)
- **Paperspace Gradient**: Free M4000 tier


In [ ]:
# ── 0. Install (run once) ─────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    'torch==2.11.0',
    'mlflow==3.11.1',
    'scikit-learn==1.8.0',
    'pandas==3.0.2',
    'numpy==2.2.5',
    'matplotlib',
    'tqdm',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('✓ Dependencies installed')

In [ ]:
# ── 1. Imports & Config ───────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mlflow
from sklearn.metrics import roc_auc_score, average_precision_score

from src.model   import DLRM
from src.dataset import (
    generate_synthetic_data, split_data, make_loaders,
    VOCAB_SIZES, NUM_DENSE, NUM_SPARSE, SPARSE_KEYS
)
from src.trainer import train, evaluate

CFG = {
    # Model
    'embedding_dim':   16,
    'bottom_mlp_dims': [64, 16],      # must end at embedding_dim
    'top_mlp_dims':    [256, 128, 64],
    'dropout':         0.1,
    # Data
    'n_samples':       500_000,
    'pos_weight':      5.0,           # handles class imbalance (~15% positive rate)
    # Training
    'batch_size':      4096,
    'lr':              1e-3,
    'weight_decay':    1e-5,
    'epochs':          10,
    'seed':            42,
    'mlflow_experiment': 'dlrm_reranker',
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## 2. Synthetic Data (Criteo-Style)
13 dense features + 6 sparse categorical features, 500K rows.
This matches the Databricks DLRM notebook schema exactly.

In [ ]:
df = generate_synthetic_data(n_samples=CFG['n_samples'], seed=CFG['seed'])
print('\nFeature schema:')
print(f'  Dense  ({NUM_DENSE}): dense_0 … dense_{NUM_DENSE-1}')
print(f'  Sparse ({NUM_SPARSE}): {SPARSE_KEYS}')
print(f'\nVocab sizes:')
for k, v in VOCAB_SIZES.items():
    print(f'  {k:<20}: {v:,}')
df.head()

In [ ]:
train_df, val_df, test_df = split_data(df, seed=CFG['seed'])
train_loader, val_loader, test_loader = make_loaders(
    train_df, val_df, test_df, batch_size=CFG['batch_size']
)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 3. DLRM Model
Bottom MLP projects 13 dense features → 16-dim (= `embedding_dim`).
6 sparse embedding tables each produce 16-dim vectors.
Feature interaction: dot-product between all 7 vectors (1 dense + 6 sparse) → 21 pairs.
Top MLP: (16 + 21) = 37 → 256 → 128 → 64 → 1 logit.

In [ ]:
model = DLRM(
    num_dense_features=NUM_DENSE,
    vocab_sizes=list(VOCAB_SIZES.values()),
    embedding_dim=CFG['embedding_dim'],
    bottom_mlp_dims=CFG['bottom_mlp_dims'],
    top_mlp_dims=CFG['top_mlp_dims'],
    dropout=CFG['dropout'],
).to(DEVICE)

if DEVICE.type == 'cuda':
    model = torch.compile(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

# Quick sanity-check forward pass
dummy_dense  = torch.randn(8, NUM_DENSE).to(DEVICE)
dummy_sparse = torch.randint(1, 100, (8, NUM_SPARSE)).to(DEVICE)
out = model(dummy_dense, dummy_sparse)
print(f'Output shape: {out.shape}  (should be torch.Size([8]))')

## 4. Train (MLflow 3.x tracked)
Uses:
- **AdamW** (standard for transformer-style models)
- **OneCycleLR** (fast warmup + cosine decay — good for ranking)
- **BCEWithLogitsLoss** with `pos_weight` to handle class imbalance
- **Gradient clipping** (max_norm=5.0)

In [ ]:
model = train(
    model, train_loader, val_loader,
    cfg=CFG, device=DEVICE, run_name='dlrm_v1'
)

## 5. Test Evaluation

In [ ]:
import torch.nn as nn
pos_weight = torch.tensor([CFG['pos_weight']], device=DEVICE)
loss_fn    = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
test_metrics = evaluate(model, test_loader, loss_fn, DEVICE)
print(f'Test AUC  : {test_metrics["auc"]:.4f}   (primary ranking metric)')
print(f'Test PR-AUC: {test_metrics["prauc"]:.4f}  (important for imbalanced labels)')
print(f'Test Loss : {test_metrics["loss"]:.4f}')

## 6. Feature Importance — Embedding Norms
Sparse features with higher embedding norms have learned more signal.
This is a quick interpretability proxy used in production systems.

In [ ]:
import matplotlib.pyplot as plt

# Get underlying module if torch.compile wraps it
m = model._orig_mod if hasattr(model, '_orig_mod') else model

norms = [
    m.sparse_embeddings.tables[i].weight.data.norm(dim=1).mean().item()
    for i in range(NUM_SPARSE)
]
plt.figure(figsize=(8, 4))
plt.bar(SPARSE_KEYS, norms, color='steelblue')
plt.title('Mean Embedding Norm per Sparse Feature\n(higher = more signal learned)')
plt.ylabel('Mean L2 Norm')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('embedding_norms.png', dpi=120)
plt.show()
print('Saved embedding_norms.png')

## 7. Score New Candidates (Inference)
In production the DLRM scores the top-100 candidates from the Two-Tower retrieval stage.

In [ ]:
model.eval()
# Simulate 100 retrieved candidates for 1 user
n_candidates = 100
dense_cands  = torch.randn(n_candidates, NUM_DENSE).to(DEVICE)
sparse_cands = torch.randint(1, 100, (n_candidates, NUM_SPARSE)).to(DEVICE)

with torch.no_grad():
    logits = model(dense_cands, sparse_cands)
    scores = torch.sigmoid(logits).cpu().numpy()

top10_idx = scores.argsort()[::-1][:10]
print('Top-10 candidate indices (re-ranked by DLRM):')
print(top10_idx)
print('Scores:', scores[top10_idx].round(4))

## 8. MLflow Run Summary

In [ ]:
import mlflow
runs = mlflow.search_runs(experiment_names=['dlrm_reranker'])
print(runs[['run_id','metrics.val_auc','metrics.val_prauc','metrics.best_val_auc']].head())

---
**Pipeline summary:**
```
All Items (50K)
    │
    ▼
[Two-Tower] ─ FAISS ANN ─► top-100 candidates   (fast, coarse)
    │
    ▼
[DLRM]      ─ Score each  ─► top-10 final items  (slow, fine-grained)
```
See companion project: **`two-tower-recommender`**